# 21.10 — Automated theorem proving

Resolution proves a goal by adding its negation and deriving the empty clause, leaving a checkable proof trail.

Automated theorem proving turns symbolic reasoning into proof search with checkable steps. Resolution generalizes rule explanations into contradiction, clauses, and proof DAGs.

Save a copy to Drive to edit.

In [ ]:

import itertools
import random

import matplotlib.pyplot as plt

random.seed(2110)


def negate_literal(literal):
    return literal[1:] if literal.startswith("~") else "~" + literal


def normalize_clause(clause):
    return tuple(sorted(set(clause), key=lambda item: item.replace("~", "") + item))


def is_tautology(clause):
    items = set(clause)
    return any(negate_literal(item) in items for item in items)


def resolve_pair(left, right):
    resolvents = []
    left_set = set(left)
    right_set = set(right)
    for literal in left_set:
        opposite = negate_literal(literal)
        if opposite in right_set:
            merged = (left_set - {literal}) | (right_set - {opposite})
            resolvent = normalize_clause(merged)
            if not is_tautology(resolvent):
                resolvents.append((literal, resolvent))
    return resolvents


def build_atp_ladder():
    rungs = []
    rungs.append({
        "name": "D1 tiny P implies Q proof",
        "clauses": [normalize_clause(["P"]), normalize_clause(["~P", "Q"])],
        "goal": "Q",
        "expected": True,
    })
    rungs.append({
        "name": "D2 three-rule Horn proof",
        "clauses": [normalize_clause(["A"]), normalize_clause(["~A", "B"]), normalize_clause(["~B", "C"]), normalize_clause(["~C", "D"])],
        "goal": "D",
        "expected": True,
    })
    rungs.append({
        "name": "D3 irrelevant clauses and conflict",
        "clauses": [normalize_clause(["A"]), normalize_clause(["~A", "B"]), normalize_clause(["R", "S"]), normalize_clause(["~S", "T"]), normalize_clause(["~B", "C"])],
        "goal": "C",
        "expected": True,
    })
    rungs.append({
        "name": "D4 protocol-style proof",
        "clauses": [normalize_clause(["sent"]), normalize_clause(["~sent", "received"]), normalize_clause(["~received", "checked"]), normalize_clause(["~checked", "safe"]), normalize_clause(["~tampered", "safe"]), normalize_clause(["logged", "audited"])],
        "goal": "safe",
        "expected": True,
    })
    clauses = [normalize_clause(["K0"])]
    for index in range(0, 10):
        clauses.append(normalize_clause(["~K" + str(index), "K" + str(index + 1)]))
    clauses.extend([normalize_clause(["NoiseA", "NoiseB"]), normalize_clause(["~NoiseB", "NoiseC"]), normalize_clause(["K3", "~K3", "Junk"])])
    rungs.append({
        "name": "D5 deep proof chain with tautology pruning",
        "clauses": clauses,
        "goal": "K10",
        "expected": True,
    })
    return rungs


## The concept, built once (D1)

Resolution cancels complementary literals: resolving $(P\lor Q)$ with $(\neg P\lor R)$ yields $(Q\lor R)$. For proof, rewrite $P\to Q$ as $\neg P\lor Q$, add the negated goal $(\neg Q)$, and derive the empty clause with $0$ literals.

In [ ]:

lesson_resolvents = resolve_pair(normalize_clause(["P", "Q"]), normalize_clause(["~P", "R"]))
lesson_first = lesson_resolvents[0][1]
lesson_clauses = [normalize_clause(["P"]), normalize_clause(["~P", "Q"])]
lesson_goal = "Q"
lesson_sizes = [3, 2, 1, 0]

assert lesson_first == normalize_clause(["Q", "R"])
assert lesson_sizes == [3, 2, 1, 0]

print("resolvent", lesson_first)
print("lesson trace sizes", lesson_sizes)


The prover below performs bounded resolution, stores every parent pair, prunes tautologies, and returns the proof trail once the empty clause is derived.

In [ ]:

def resolution_prove(clauses, goal, max_steps=200):
    initial = [normalize_clause(clause) for clause in clauses]
    support = initial + [normalize_clause([negate_literal(goal)])]
    known = set(support)
    parents = {clause: None for clause in support}
    steps = []
    for step in range(max_steps):
        pairs = list(itertools.combinations(list(known), 2))
        added = False
        for left, right in pairs:
            for pivot, resolvent in resolve_pair(left, right):
                if resolvent in known:
                    continue
                known.add(resolvent)
                parents[resolvent] = (left, right, pivot)
                steps.append({
                    "step": len(steps) + 1,
                    "left": left,
                    "right": right,
                    "pivot": pivot,
                    "resolvent": resolvent,
                })
                added = True
                if len(resolvent) == 0:
                    return True, steps, parents, known
        if not added:
            return False, steps, parents, known
    return False, steps, parents, known

proved, steps, parents, known = resolution_prove(lesson_clauses, lesson_goal)
empty_steps = [step for step in steps if len(step["resolvent"]) == 0]

assert proved is True
assert len(empty_steps[-1]["resolvent"]) == 0

print("proved", proved)
print("steps", len(steps))
print("last resolvent", empty_steps[-1]["resolvent"])


## The dataset ladder

D1–D5 grow from a one-rule proof to a deep D5 chain with irrelevant clauses and tautology pruning.

In [ ]:

atp_rungs = build_atp_ladder()

for rung in atp_rungs:
    print(rung["name"])
    print("  clauses", len(rung["clauses"]))
    print("  goal", rung["goal"])
    print("  sample", rung["clauses"][:3])


## Run the same method across D1–D5

In [ ]:

atp_results = []

for rung in atp_rungs:
    proved, steps, parents, known = resolution_prove(rung["clauses"], rung["goal"])
    assert proved == rung["expected"]
    atp_results.append({
        "name": rung["name"],
        "clauses": len(rung["clauses"]),
        "proved": proved,
        "steps": len(steps),
        "known": len(known),
        "steps_detail": steps,
    })

print("rung | proved | clauses | resolution steps | known clauses")
for item in atp_results:
    print(item["name"], item["proved"], item["clauses"], item["steps"], item["known"])


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for index, item in enumerate(atp_results):
    ax = axes[0][index]
    ax.set_title("D" + str(index + 1))
    ax.axis("off")
    lines = []
    for step in item["steps_detail"][:6]:
        lines.append(str(step["step"]) + ": " + str(step["left"]) + " + " + str(step["right"]) + " -> " + str(step["resolvent"]))
    ax.text(0.02, 0.95, "\n".join(lines), va="top", family="monospace", fontsize=7)

x_values = list(range(1, 6))
steps_values = [item["steps"] for item in atp_results]
clauses_values = [item["clauses"] for item in atp_results]
axes[1][0].plot(x_values, steps_values, marker="o", label="resolution steps")
axes[1][0].plot(x_values, clauses_values, marker="s", label="input clauses")
axes[1][0].set_xticks(x_values)
axes[1][0].set_xlabel("rung")
axes[1][0].set_ylabel("count")
axes[1][0].set_title("proof steps vs size")
axes[1][0].legend()
for ax in axes[1][1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: ignoring explanation paths. A theorem prover should not just say proved; it should show which parent clauses produced every resolvent.

In [ ]:

d5 = atp_rungs[-1]
proved, steps, parents, known = resolution_prove(d5["clauses"], d5["goal"])
empty = normalize_clause([])

print("proved without explanation would be", proved)
print("stored parent for empty clause", parents.get(empty))
print("last five proof steps")
for step in steps[-5:]:
    print(step)


## Evaluate it + Practice

- Metric: proof correctness, resolution steps, empty-clause reached.
- No-skill baseline: claim the goal from a forward chain without contradiction.
- Cheap sanity check: D1 derives the empty clause with 0 literals.
- Ablation: drop parent storage and the proof becomes unauditable.
- Failure signals: proved=True appears without a parent trace.

Practice 1: Change the D2 instance by one constraint and predict the metric before running.

Practice 2: Add one contradictory or noisy condition to D4 and explain how the trace changes.

Practice 3: On D5, record the first step where pruning or explanation prevents a wrong answer.